In [1]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

ImportError: cannot import name 'display' from 'IPython.core.display' (C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\IPython\core\display.py)

In [ ]:
# got an error here, so running the next cell's code instead

In [2]:
from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [4]:
%pip install pandas matplotlib scikit-learn

     ---------------------------------------- 0.0/52.8 kB ? eta -:--:--
     ---------------------------------------- 0.0/52.8 kB ? eta -:--:--
     -------------------------------------- - 51.2/52.8 kB ? eta -:--:--
     -------------------------------------- - 51.2/52.8 kB ? eta -:--:--
     -------------------------------------- - 51.2/52.8 kB ? eta -:--:--
     -------------------------------------- - 51.2/52.8 kB ? eta -:--:--
     -------------------------------------- 52.8/52.8 kB 170.4 kB/s eta 0:00:00
     ---------------------------------------- 0.0/119.8 kB ? eta -:--:--
     ----------------------------------- -- 112.6/119.8 kB 6.4 MB/s eta 0:00:01
     -------------------------------------- 119.8/119.8 kB 2.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/9.9 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.9 MB 5.7 MB/s eta 0:00:02
   -- ------------------------------------- 0.5/9.9 MB 4.7 MB/s eta 0:00:02
   --- -----------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.0 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [8]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("../data/kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


,text,label
0,"DEAR SIR, STRICTLY A PRIVATE BUSINESS PROPOSAL...",1
1,Will do.,0
2,Nora--Cheryl has emailed dozens of memos about...,0
3,Dear Sir=2FMadam=2C I know that this proposal ...,1
4,fyi,0
...,...,...
995,So what's the latest? It sounds contradictory ...,0
996,"TRANSFER OF 36,759,000.00 MILLION POUNDS TO YO...",1
997,Barb I will call to explain. Are you back in t...,0
998,Yang on travelNot free tonite.May work tomorrow,0


### Let's divide the training and test set into two partitions

In [10]:
print(data.columns)

Index(['text', 'label'], dtype='str')


In [11]:
from sklearn.model_selection import train_test_split

X = data.drop(columns=["label"])
y = data["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

(800, 1) (200, 1)
(800,) (200,)


## Data Preprocessing

In [12]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [22]:
import re

def remove_html_code(text):
    text = str(text)

    # Removing inline JavaScript blocks: <script> ... </script>:
    text = re.sub(r"<script.*?>.*?</script>", " ", text, flags=re.DOTALL | re.IGNORECASE)

    # Removing inline CSS/style blocks: <style> ... </style>:
    text = re.sub(r"<style.*?>.*?</style>", " ", text, flags=re.DOTALL | re.IGNORECASE)

    # Removing HTML comments: <!-- comment -->
    # This is done before removing regular tags because comments may contain > characters:
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # Removing remaining HTML tags such as <p>, <a>, <br>, <div>:
    text = re.sub(r"<[^>]+>", " ", text)

    return text

- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [25]:
import string

# Removing noisy characters and standardising the text:

def clean_text_basic(text):
    text = str(text)

    # Removing all special characters and punctuation
    text = re.sub(f"[{re.escape(string.punctuation)}]", " ", text)

    # Removing numbers
    text = re.sub(r'\d+', ' ', text)

    # Removing all single characters
    text = re.sub(r'\b[a-zA-Z]\b', ' ', text)

    # Removing single characters from the start
    text = re.sub(r'^[a-zA-Z]\s+', ' ', text)

    # Removing prefix lowercase b, often from byte strings like b'hello'
    text = re.sub(r"^b\s+", " ", text)

    # Substituting multiple spaces with one space
    text = re.sub(r'\s+', ' ', text)

    # Converting to lowercase
    text = text.lower()

    # Removing leading/trailing spaces
    text = text.strip()

    return text

In [26]:
X_train_no_html = X_train.apply(remove_html_code)
X_test_no_html = X_test.apply(remove_html_code)

In [27]:
X_train_clean = X_train_no_html.apply(clean_text_basic)
X_test_clean = X_test_no_html.apply(clean_text_basic)

In [28]:
print(X_train.iloc[0])
print("----- After HTML cleaning -----")
print(X_train_no_html.iloc[0])
print("----- After Basic cleaning -----")
print(X_train_clean.iloc[0])

text    Dear=2C Good day hope fine=2Cdear am writting ...
Name: 442, dtype: str
----- After HTML cleaning -----
442    Dear=2C Good day hope fine=2Cdear am writting ...
962    FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...
971                                             Will do.
190    FROM THE DESK OF DR.ADAMU  ISMALERAUDITING AND...
551    Dear Friend, My name is LOI C.ESTRADA,The wife...
                             ...                        
625       Dear, &nbsp;  It is my ple...
728    Will do. Think we are in good shape. Immelt In...
664    With your permission we are changing the state...
798                      Yes we will. Thank you Courtney
Name: text, Length: 800, dtype: str
----- After Basic cleaning -----
dear good day hope fine cdear am writting from mr henry kaborethe chief auditor incharge will do from the desk of dr adamu ismalerauditing and dear friend my name is loi estrada the wife dear nbsp it is my ple will do think we are in good shape immelt in with your p

## Now let's work on removing stopwords
Remove the stopwords.

In [29]:
import nltk
from nltk.corpus import stopwords

# Download stopwords list if needed
nltk.download("stopwords")

# Load English stopwords
stop_words = set(stopwords.words("english"))

def remove_stopwords(text):
    words = text.split()
    words_no_stopwords = [word for word in words if word not in stop_words]
    return " ".join(words_no_stopwords)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [30]:
X_train_no_stopwords = X_train_clean.apply(remove_stopwords)
X_test_no_stopwords = X_test_clean.apply(remove_stopwords)

In [31]:
print(X_train_clean.iloc[0])
print("----- WITHOUT STOPWORDS -----")
print(X_train_no_stopwords.iloc[0])

dear good day hope fine cdear am writting from mr henry kaborethe chief auditor incharge will do from the desk of dr adamu ismalerauditing and dear friend my name is loi estrada the wife dear nbsp it is my ple will do think we are in good shape immelt in with your permission we are changing the state yes we will thank you courtney name text length dtype str
----- WITHOUT STOPWORDS -----
dear good day hope fine cdear writting mr henry kaborethe chief auditor incharge desk dr adamu ismalerauditing dear friend name loi estrada wife dear nbsp ple think good shape immelt permission changing state yes thank courtney name text length dtype str


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [32]:
from nltk.stem import WordNetLemmatizer

nltk.download("wordnet")
nltk.download("omw-1.4")

lemmatizer = WordNetLemmatizer()

def lemmatize_text(text):
    words = text.split()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(lemmatized_words)

[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\ASUS\AppData\Roaming\nltk_data...


In [33]:
X_train_lemmatized = X_train_no_stopwords.apply(lemmatize_text)
X_test_lemmatized = X_test_no_stopwords.apply(lemmatize_text)

In [34]:
print(X_train_no_stopwords.iloc[0])
print("----- LEMMATIZED -----")
print(X_train_lemmatized.iloc[0])

dear good day hope fine cdear writting mr henry kaborethe chief auditor incharge desk dr adamu ismalerauditing dear friend name loi estrada wife dear nbsp ple think good shape immelt permission changing state yes thank courtney name text length dtype str
----- LEMMATIZED -----
dear good day hope fine cdear writting mr henry kaborethe chief auditor incharge desk dr adamu ismalerauditing dear friend name loi estrada wife dear nbsp ple think good shape immelt permission changing state yes thank courtney name text length dtype str


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [ ]:
from collections import Counter
import pandas as pd

# Combining the cleaned/lemmatized training text with the training labels.
# reset_index(drop=True) makes sure both Series align row by row.
eda_data = pd.DataFrame({
    "text": X_train_lemmatized.reset_index(drop=True),
    "label": y_train.reset_index(drop=True)
})

# Quick check
print(eda_data.head())
print(eda_data["label"].value_counts())

                                                text  label
0  dear good day hope fine cdear writting mr henr...      1
1                                                NaN      1
2                                                NaN      0
3                                                NaN      1
4                                                NaN      1
label
0    446
1    354
Name: count, dtype: int64


In [38]:
# Making sure text is string and missing values are removed/replaced:
eda_data["text"] = eda_data["text"].fillna("").astype(str)

# Separating messages by class:
ham_messages = eda_data[eda_data["label"] == 0]["text"]
spam_messages = eda_data[eda_data["label"] == 1]["text"]

# Joining all messages in each class into one long string:
ham_text = " ".join(ham_messages)
spam_text = " ".join(spam_messages)

In [39]:
print(type(ham_text))
print(type(spam_text))

print("Ham text sample:")
print(ham_text[:500])

print("\nSpam text sample:")
print(spam_text[:500])

<class 'str'>
<class 'str'>
Ham text sample:
                                                                                                                                                                                                                                                                                                                                                                                                                                                             

Spam text sample:
dear good day hope fine cdear writting mr henry kaborethe chief auditor incharge desk dr adamu ismalerauditing dear friend name loi estrada wife dear nbsp ple think good shape immelt permission changing state yes thank courtney name text length dtype str                                                                                                                                                                                                                                            

In [ ]:
from collections import Counter

# Counting word frequency for ham and spam
ham_word_counts = Counter(ham_text.split())
spam_word_counts = Counter(spam_text.split())

In [42]:
print("Number of ham words:", len(ham_text.split()))
print("Number of spam words:", len(spam_text.split()))

print("Top 10 ham words:")
print(ham_word_counts.most_common(10))

print("\nTop 10 spam words:")
print(spam_word_counts.most_common(10))

Number of ham words: 0
Number of spam words: 41
Top 10 ham words:
[]

Top 10 spam words:
[('dear', 3), ('good', 2), ('name', 2), ('day', 1), ('hope', 1), ('fine', 1), ('cdear', 1), ('writting', 1), ('mr', 1), ('henry', 1)]


## Extra features

In [ ]:
# Creating train and validation DataFrames using our cleaned/lemmatized text
data_train = pd.DataFrame({
    "preprocessed_text": X_train_lemmatized.reset_index(drop=True),
    "label": y_train.reset_index(drop=True)
})

data_val = pd.DataFrame({
    "preprocessed_text": X_test_lemmatized.reset_index(drop=True),
    "label": y_test.reset_index(drop=True)
})

In [ ]:
# Forcing every value to be text and removing missing-value problems:

data_train["preprocessed_text"] = data_train["preprocessed_text"].fillna("").astype(str)
data_val["preprocessed_text"] = data_val["preprocessed_text"].fillna("").astype(str)

# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list, na = False)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words, na = False)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list, na = False)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words, na = False)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,preprocessed_text,label,money_mark,suspicious_words,text_len
0,dear good day hope fine cdear writting mr henr...,1,0,0,254
1,,1,0,0,0
2,,0,0,0,0
3,,1,0,0,0
4,,1,0,0,0


## How would work the Bag of Words with Count Vectorizer concept?

In [ ]:
# Applying the CountVectorizer concept:

from sklearn.feature_extraction.text import CountVectorizer

# Creating CountVectorizer
count_vectorizer = CountVectorizer()

# Fitting on training text and transforming training text into word-count vectors
X_train_count = count_vectorizer.fit_transform(data_train["preprocessed_text"])

# Transforming validation text using the vocabulary learned from training text
X_val_count = count_vectorizer.transform(data_val["preprocessed_text"])

# Quick check
print(X_train_count.shape)
print(X_val_count.shape)

(800, 37)
(200, 37)


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [48]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Create TF-IDF vectorizer
tfidf_vectorizer = TfidfVectorizer()

# Fit on training text and transform training text into TF-IDF vectors
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])

# Transform validation text using the vocabulary/weights learned from training text
X_val_tfidf = tfidf_vectorizer.transform(data_val["preprocessed_text"])

# Print the shape of the vectorized datasets
print(X_train_tfidf.shape)
print(X_val_tfidf.shape)

(800, 37)
(200, 37)


## And the Train a Classifier?

In [ ]:
from sklearn.linear_model import LogisticRegression

# Creating the classifier/model
classifier = LogisticRegression(max_iter=1000)

# Training the classifier on the training data
classifier.fit(X_train_tfidf, data_train["label"])

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Predicting labels for validation data
y_pred = classifier.predict(X_val_tfidf)

# Evaluating performance
print("Accuracy:", accuracy_score(data_val["label"], y_pred))
print("\nClassification report:")
print(classification_report(data_val["label"], y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(data_val["label"], y_pred))

Accuracy: 0.565

Classification report:
              precision    recall  f1-score   support

           0       0.56      1.00      0.72       112
           1       1.00      0.01      0.02        88

    accuracy                           0.56       200
   macro avg       0.78      0.51      0.37       200
weighted avg       0.76      0.56      0.41       200


Confusion matrix:
[[112   0]
 [ 87   1]]


In [ ]:
# The model has basically learned how to predict almost everything as ham.

# That gives mediocre accuracy, but it fails the real goal, which is to detect spam.

# In summyary, acuracy alone is not enough. We need recall/F1 for spam detection too.

### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [ ]:
# Your code